# *Rhizopus* isolate R144 — statistics and figures
Run the cells in order (*Runtime → Run all*). Upload `Tur_compiled_analysis_ready.xlsx` when asked.

## 1. Setup

In [ ]:
# =====================================================================
# CELL 1 — Setup: fonts, libraries, data and figure style
# =====================================================================

# Install Times New Roman (Microsoft core fonts) and Liberation Serif (fallback)
!echo "ttf-mscorefonts-installer msttcorefonts/accept-mscorefonts-eula select true" | debconf-set-selections
!apt-get -qq install -y ttf-mscorefonts-installer fonts-liberation > /dev/null 2>&1

import io, os, glob, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats
from scipy.optimize import curve_fit
from PIL import Image
from google.colab import files

# ---- Load data (upload the Excel file if it is not already in Colab) ----
DATA = "Tur_compiled_analysis_ready.xlsx"
if not os.path.exists(DATA):
    DATA = next(iter(files.upload()))

temp = pd.read_excel(DATA, "Temperature_Growth")   # colony diameter at 10–40 °C
ph   = pd.read_excel(DATA, "pH_Growth")            # colony diameter at pH 3–8
enz  = pd.read_excel(DATA, "Enzyme_Activity")      # enzyme activity (U/mL)
les  = pd.read_excel(DATA, "Lesion_Size")          # discolored-zone size in trees

enz["Enzyme"] = enz["Enzyme"].str.strip().replace({"Ligninase": "Lignin peroxidase"})
les["LW_ratio"] = les["Length_cm"] / les["Width_cm"]   # length:width ratio of each zone

# ---- Font: Times New Roman if installed, otherwise Liberation Serif ----
for f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf") + glob.glob("/usr/share/fonts/truetype/liberation/*.ttf"):
    font_manager.fontManager.addfont(f)
names = {f.name for f in font_manager.fontManager.ttflist}
FONT = "Times New Roman" if "Times New Roman" in names else "Liberation Serif"

# ---- Colours (flat-UI palette) and global figure style ----
BLUE, RED, GREEN, ORANGE, PURPLE, NAVY = "#3498DB", "#E74C3C", "#27AE60", "#F39C12", "#8E44AD", "#1B2631"
GRID, AXIS = "#DCDCDC", "#4D4D4D"
plt.rcParams.update({
    "font.family": FONT, "font.size": 12, "font.weight": "normal",
    "axes.titlesize": 12, "axes.titleweight": "normal", "axes.labelsize": 12, "axes.labelweight": "normal",
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
    "figure.titlesize": 12, "figure.titleweight": "normal",
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "xtick.color": AXIS, "ytick.color": AXIS,
    "axes.labelcolor": "black", "xtick.labelcolor": "black", "ytick.labelcolor": "black",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": True, "legend.edgecolor": "#BFBFBF", "legend.facecolor": "#F2F2F2", "legend.framealpha": 1,
    "mathtext.fontset": "custom", "mathtext.rm": FONT, "mathtext.it": f"{FONT}:italic", "mathtext.cal": FONT,
    "pdf.fonttype": 42, "savefig.dpi": 600, "figure.dpi": 110,
})
W = 190 / 25.4                     # figure width: 190 mm in inches
OUT = "figures"                    # output folder for all figures
os.makedirs(OUT, exist_ok=True)
VAL = 10.5                         # font size of value labels on bars
EB = dict(ecolor="black", elinewidth=0.6, capsize=3, capthick=0.6)   # error-bar style
rng = np.random.default_rng(1)     # fixed seed so point jitter is identical every run

def save(fig, name):
    """Save a figure as PNG (600 dpi), PDF (embedded fonts) and TIFF (600 dpi, RGB, LZW), then show it."""
    fig.savefig(f"{OUT}/{name}.png", dpi=600)
    fig.savefig(f"{OUT}/{name}.pdf")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=600)
    Image.open(buf).convert("RGB").save(f"{OUT}/{name}.tif", dpi=(600, 600), compression="tiff_lzw")
    plt.show()

# ---- Output of this cell: font and a short overview of every data sheet ----
print(f"Font used: {FONT}\n")
for name, df, g in [("Temperature growth", temp, "Temperature_C"), ("pH growth", ph, "pH"),
                    ("Enzyme activity", enz, "Enzyme"), ("Discolored zones", les, "DAI")]:
    print(f"{name}: {len(df)} rows, columns = {list(df.columns)}")
    print(f"  replicates per group: {df.groupby(g, sort=False).size().to_dict()}")
    print(df.head(3).to_string(index=False), "\n")

## 2. Statistical analysis (tests, letters and Figure S3)

In [ ]:
# =====================================================================
# CELL 2 — Statistical analysis: assumption checks, ANOVA, post hoc tests, letters
# =====================================================================
# Workflow for every variable (e.g. colony diameter across temperatures):
#   Step 1. Residuals: subtract each group's mean from its values.
#   Step 2. Shapiro–Wilk test on the residuals -> are the errors normally distributed?
#   Step 3. Brown–Forsythe test -> are the variances (spread) equal across groups?
#   Step 4. Equal variances   -> one-way ANOVA + Tukey HSD
#           Unequal variances -> Welch ANOVA + Games–Howell
#   Step 5. Compact letter display: groups sharing a letter do not differ (p >= 0.05).
#   Step 6. Supplementary Figure S3: the same assumption checks shown as plots.

def letters(names, means, sig):
    """Compact letter display (letters above the bars).
    names = group names, means = group means, sig = list of pairs that differ significantly.
    The group with the highest mean gets 'a'."""
    order = [n for _, n in sorted(zip(means, names), reverse=True)]   # groups from highest to lowest mean
    cols = [set(order)]                      # start: all groups share one letter
    for a, b in sig:                         # a significant pair cannot share a letter -> split that letter group
        new = []
        for c in cols:
            new += [c - {a}, c - {b}] if (a in c and b in c) else [c]
        new = [c for c in new if c]
        cols = []
        for c in new:                        # remove letter groups that are contained in another, and duplicates
            if not any(c < o for o in new) and c not in cols:
                cols.append(c)
    cols.sort(key=lambda c: min(order.index(n) for n in c))           # 'a' goes to the highest means
    lab = {n: "" for n in names}
    for i, c in enumerate(cols):
        for n in c:
            lab[n] += "abcdefghij"[i]
    return lab

def compare(df, g, y):
    """One-way comparison of variable y across groups g.
    Prints the test results and returns a table of mean, SE and letter for each group."""
    names = list(dict.fromkeys(df[g]))                          # group names in data order
    grp = [df.loc[df[g] == n, y].to_numpy() for n in names]     # list of arrays, one per group
    k, n_i = len(grp), np.array([len(x) for x in grp])          # number of groups, replicates per group
    m, v = np.array([x.mean() for x in grp]), np.array([x.var(ddof=1) for x in grp])   # group means and variances

    # ---- Step 1: residuals ----
    # ANOVA model: value = group mean + random error (residual).
    # The normality assumption applies to these errors, NOT to the raw data:
    # raw data from groups with different means (e.g. 15 mm at 10 °C, 90 mm at 30 °C) pooled together
    # never look normal, even if every group is perfectly normal. Subtracting each group's mean
    # removes the treatment effect and leaves only the random scatter that ANOVA assumes is normal.
    resid = np.concatenate([x - x.mean() for x in grp])

    # ---- Step 2: Shapiro–Wilk normality test on the residuals ----
    # p >= 0.05 -> residuals can be treated as normal (assumption met).
    # p <  0.05 -> residuals depart from normality; ANOVA is fairly robust with n >= 10 per group,
    #              and a Kruskal–Wallis test (in R144_analysis.py) is used as a check.
    # All residuals are pooled (21–30 values) because n = 3 per group is too small to test groups separately.
    sw_p = stats.shapiro(resid).pvalue

    # ---- Step 3: Brown–Forsythe test for equal variances ----
    # Levene's test with center="median" = Brown–Forsythe test. Using the median instead of the mean
    # makes it reliable even when data are not normal (unlike Bartlett's test, which gives many
    # false "unequal variance" results with non-normal data).
    # p >= 0.05 -> variances can be treated as equal; p < 0.05 -> variances differ.
    lev_p = stats.levene(*grp, center="median").pvalue

    # ---- Step 4: choose the test according to Step 3 ----
    if lev_p >= 0.05:
        # Equal variances: classic one-way ANOVA, then Tukey HSD for all pairs of groups
        F, p = stats.f_oneway(*grp)
        df1, df2, test = k - 1, n_i.sum() - k, "ANOVA + Tukey HSD"
        P = stats.tukey_hsd(*grp).pvalue          # matrix of pairwise p-values, P[i, j]
        pair = lambda i, j: P[i, j]
    else:
        # Unequal variances: Welch ANOVA (weights each group by n / variance)
        w = n_i / v
        mw = (w * m).sum() / w.sum()              # weighted grand mean
        t = ((1 - w / w.sum()) ** 2 / (n_i - 1)).sum()
        F = ((w * (m - mw) ** 2).sum() / (k - 1)) / (1 + 2 * (k - 2) / (k ** 2 - 1) * t)
        df1, df2, test = k - 1, (k ** 2 - 1) / (3 * t), "Welch ANOVA + Games-Howell"
        p = stats.f.sf(F, df1, df2)
        def pair(i, j):
            # Games–Howell post hoc test (does not assume equal variances).
            # s = squared standard error of the difference; dfw = Welch–Satterthwaite degrees of freedom.
            # q = |difference| / sqrt(s / 2) follows the studentized range distribution (q = t × √2).
            s = v[i] / n_i[i] + v[j] / n_i[j]
            dfw = s ** 2 / ((v[i] / n_i[i]) ** 2 / (n_i[i] - 1) + (v[j] / n_i[j]) ** 2 / (n_i[j] - 1))
            return stats.studentized_range.sf(abs(m[i] - m[j]) / np.sqrt(s / 2), k, dfw)

    # ---- Step 5: significant pairs -> letters ----
    sig = [(names[i], names[j]) for i in range(k) for j in range(i + 1, k) if pair(i, j) < 0.05]
    out = df.groupby(g, sort=False)[y].agg(mean="mean", se="sem")   # se = standard error of the mean
    out["letter"] = [letters(names, m, sig)[n] for n in out.index]

    # ---- Print results for this variable ----
    print(f"{y} by {g}: {test}")
    print(f"  F({df1}, {df2:.1f}) = {F:.2f}, p = {p:.2e} | Shapiro–Wilk p = {sw_p:.3f}, Brown–Forsythe p = {lev_p:.3f}")
    print(out.round(2).to_string(), "\n")
    return out

# Run the comparison for every response variable
st_t = compare(temp, "Temperature_C", "Radial_Growth_mm")    # growth vs temperature
st_p = compare(ph, "pH", "Radial_Growth_mm")                 # growth vs pH
st_e = compare(enz, "Enzyme", "Activity_U_per_ml")           # activity among enzymes
st_l = {v: compare(les, "DAI", v) for v in ["Length_cm", "Width_cm", "Area_cm2", "LW_ratio"]}   # zones vs days

# ---- Step 6: Supplementary Figure S3 — picture of the assumption checks (Steps 2 and 3) ----
# Left: normal Q–Q plot of the residuals (points near the red line = normal; matches Shapiro–Wilk).
# Right: residuals vs group mean (similar vertical spread in every group = equal variances; matches Brown–Forsythe).
sets = [(temp, "Temperature_C", "Radial_Growth_mm", "Temperature"), (ph, "pH", "Radial_Growth_mm", "pH"),
        (enz, "Enzyme", "Activity_U_per_ml", "Enzymes"), (les, "DAI", "Area_cm2", "Discolored area")]
fig, axs = plt.subplots(4, 2, figsize=(W, 9.6), constrained_layout=True)
for r, (df, g, y, name) in enumerate(sets):
    fitted = df.groupby(g)[y].transform("mean")
    res = df[y] - fitted                                         # residual = value − its group mean
    (osm, osr), (slope, icpt, _) = stats.probplot(res)
    a, b = axs[r]
    a.scatter(osm, osr, s=14, color=BLUE, edgecolor="black", lw=0.3, zorder=3)
    a.plot(osm, slope * osm + icpt, color=RED, lw=1)
    a.set(title=f"({'abcdefgh'[2 * r]}) {name}: normal Q–Q", xlabel="Theoretical quantiles", ylabel="Residual")
    b.scatter(fitted, res, s=14, color=GREEN, edgecolor="black", lw=0.3, zorder=3)
    b.axhline(0, color=AXIS, lw=0.8, ls="--")
    b.set(title=f"({'abcdefgh'[2 * r + 1]}) {name}: residuals vs fitted", xlabel="Fitted value (group mean)", ylabel="Residual")
fig.suptitle("Residual diagnostics")
save(fig, "FigureS3_residual_diagnostics")
print("Cell 2 finished: statistics done, Figure S3 saved")


## 3. Growth models — select the best model, report only that one

In [ ]:
# =====================================================================
# CELL 3 — Growth models: choose the best model, then report only that one
# =====================================================================
# Step 1  Define the candidate models.
# Step 2  Fit every candidate to the same data and rank them by AICc.
# Step 3  Keep the model with the lowest AICc (the best model).
# Step 4  Report only the best model: parameters with 95% CI, R², RMSE, residual normality.
# The best model is stored in best_T and best_p and is used for Figure 3 (Cell 5).

# ---------------- Step 1: candidate models ----------------
def ctmi(T, mu, tmin, topt, tmax):
    """Cardinal temperature model with inflection (Rosso et al. 1993)."""
    T = np.asarray(T, float)
    y = mu * (T - tmax) * (T - tmin) ** 2 / ((topt - tmin) * ((topt - tmin) * (T - topt) - (topt - tmax) * (topt + tmin - 2 * T)))
    return np.where((T > tmin) & (T < tmax), y, 0)

def beta_m(T, mu, tmin, topt, tmax):
    """Beta function (Yan & Hunt 1999): same four parameters as CTMI, different curve shape."""
    T = np.asarray(T, float)
    a = (topt - tmin) / (tmax - topt)
    y = mu * np.clip((tmax - T) / (tmax - topt), 0, None) * np.clip((T - tmin) / (topt - tmin), 0, None) ** a
    return np.where((T > tmin) & (T < tmax), y, 0)

def briere2(T, a, tmin, tmax, m):
    """Briere-2 model (Briere et al. 1999)."""
    T = np.asarray(T, float)
    return np.where((T > tmin) & (T < tmax), a * T * (T - tmin) * np.clip(tmax - T, 0, None) ** (1 / m), 0)

def gauss(x, mu, xopt, s):
    """Gaussian (bell-shaped) pH response: mu = growth at the optimum, xopt = optimum, s = curve width."""
    return mu * np.exp(-(np.asarray(x, float) - xopt) ** 2 / (2 * s ** 2))

def cpm(x, mu, pmin, popt, pmax):
    """Cardinal pH model (Rosso et al. 1995)."""
    x = np.asarray(x, float)
    num = (x - pmin) * (x - pmax)
    return np.where((x > pmin) & (x < pmax), mu * num / (num - (x - popt) ** 2), 0)

def quadratic(x, a, b, c):
    """Second-order polynomial (empirical, symmetric)."""
    x = np.asarray(x, float)
    return a + b * x + c * x ** 2

# Each candidate: name, function, starting values, bounds, parameter names.
# Bounds keep the lower limit below the lowest tested level and the upper limit above the highest,
# so every observation stays inside the growth range during fitting.
lim_T = ([10, -10, 20, 40.5], [150, 9.9, 39, 60])
cand_T = [("CTMI", ctmi, [90, 5, 29, 42], lim_T, ["µopt", "Tmin", "Topt", "Tmax"]),
          ("Beta", beta_m, [90, 5, 29, 42], lim_T, ["µopt", "Tmin", "Topt", "Tmax"]),
          ("Briere-2", briere2, [0.01, 3, 42, 2], ([1e-6, -10, 40.5, 0.5], [1, 9.9, 60, 10]), ["a", "Tmin", "Tmax", "m"]),
          ("Quadratic", quadratic, [0, 1, -0.01], (-np.inf, np.inf), ["a", "b", "c"])]
lim_P = ([10, 0, 4.1, 8.05], [150, 2.99, 7.9, 14])
cand_P = [("Gaussian", gauss, [88, 5.7, 1.3], ([10, 4, 0.2], [150, 8, 5]), ["µopt", "pHopt", "sigma"]),
          ("CPM", cpm, [88, 2.5, 6, 9], lim_P, ["µopt", "pHmin", "pHopt", "pHmax"]),
          ("Beta", beta_m, [88, 2.5, 6, 9], lim_P, ["µopt", "pHmin", "pHopt", "pHmax"]),
          ("Quadratic", quadratic, [0, 30, -3], (-np.inf, np.inf), ["a", "b", "c"])]

# ---------------- Step 2 + 3: rank by AICc and keep the best ----------------
def select_best(cands, x, y, label):
    """Fit all candidates, print a short AICc ranking and return the best (lowest AICc) model."""
    rows = []
    for name, f, p0, bounds, names in cands:
        p, cov = curve_fit(f, x, y, p0=p0, bounds=bounds, maxfev=200000)
        n, k = len(y), len(p)
        rss = ((y - f(x, *p)) ** 2).sum()                                      # residual sum of squares
        aicc = n * np.log(rss / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1)    # small-sample AIC
        rows.append(dict(Model=name, k=k, AICc=aicc, f=f, p=p, cov=cov, names=names))
    tab = pd.DataFrame(rows).sort_values("AICc").reset_index(drop=True)
    tab["dAICc"] = tab["AICc"] - tab["AICc"].min()                            # 0 for the best model
    tab["Weight"] = np.exp(-tab["dAICc"] / 2) / np.exp(-tab["dAICc"] / 2).sum()   # Akaike weight
    print(f"{label}: model ranking (lowest AICc = best)")
    print(tab[["Model", "k", "AICc", "dAICc", "Weight"]].round(3).to_string(index=False))
    best = tab.iloc[0]
    close = tab[(tab["dAICc"] > 0) & (tab["dAICc"] < 2)]["Model"].tolist()
    print(f"-> Best model: {best['Model']}"
          + (f"  (note: {', '.join(close)} within ΔAICc < 2, i.e. similar support)" if close else ""), "\n")
    return best

# ---------------- Step 4: report only the best model ----------------
def report(best, x, y, grid, opt_name):
    """Print the best model's fit statistics and parameters (95% CI) and return a summary dict."""
    f, p, cov, names = best["f"], best["p"], best["cov"], best["names"]
    t = stats.t.ppf(0.975, len(y) - len(p))
    ci = t * np.sqrt(np.diag(cov))                                     # 95% CI half-width = t × SE
    resid = y - f(x, *p)                                              # observed − fitted
    r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
    rmse = np.sqrt((resid ** 2).mean())
    sw_p = stats.shapiro(resid).pvalue                                # residual normality (should be > 0.05)
    print(f"{best['Model']} model: R² = {r2:.3f}, RMSE = {rmse:.2f} mm, residual Shapiro–Wilk p = {sw_p:.3f}")
    for n, a, c in zip(names, p, ci):
        print(f"  {n:6s} = {a:8.3f}   95% CI {a - c:.3f} – {a + c:.3f}")
    if opt_name in names:                                             # optimum is a model parameter
        i = names.index(opt_name)
        opt, lo, hi = p[i], p[i] - ci[i], p[i] + ci[i]
    else:                                                             # optimum = peak of the curve; CI by simulation
        draws = np.random.default_rng(1).multivariate_normal(p, cov, 2000)
        peaks = [grid[np.argmax(f(grid, *d))] for d in draws]
        opt, (lo, hi) = grid[np.argmax(f(grid, *p))], np.percentile(peaks, [2.5, 97.5])
        print(f"  optimum = {opt:.3f}   95% CI {lo:.3f} – {hi:.3f}")
    print()
    return dict(name=best["Model"], f=f, p=p, opt=opt, lo=lo, hi=hi, r2=r2, rmse=rmse, sw_p=sw_p)

xT, yT = temp["Temperature_C"].to_numpy(float), temp["Radial_Growth_mm"].to_numpy(float)
xP, yP = ph["pH"].to_numpy(float), ph["Radial_Growth_mm"].to_numpy(float)

best_T = report(select_best(cand_T, xT, yT, "Temperature"), xT, yT, np.linspace(10, 40, 3001), "Topt")
best_p = report(select_best(cand_P, xP, yP, "pH"), xP, yP, np.linspace(3, 8, 3001), "pHopt")

print(f"Optimum temperature = {best_T['opt']:.2f} °C (95% CI {best_T['lo']:.2f}–{best_T['hi']:.2f}), {best_T['name']} model")
print(f"Optimum pH          = {best_p['opt']:.2f} (95% CI {best_p['lo']:.2f}–{best_p['hi']:.2f}), {best_p['name']} model")
# Tmin and Tmax lie outside the tested range (10–40 °C), so they are extrapolations.


## 4. Discolored-zone kinetics

In [ ]:
# =====================================================================
# CELL 4 — Discolored-zone kinetics: expansion rates and model comparison
# =====================================================================
dai = les["DAI"].to_numpy(float)

# Relative expansion rate (per day) = slope of ln(dimension) against days after inoculation
rl = stats.linregress(dai, np.log(les["Length_cm"]))
rw = stats.linregress(dai, np.log(les["Width_cm"]))
ra = stats.linregress(dai, np.log(les["Area_cm2"]))
t_rw = (rl.slope - rw.slope) / np.hypot(rl.stderr, rw.stderr)          # t-test: length vs width rate
print("Relative expansion rates (d⁻¹)")
print(f"  length {rl.slope:.4f}, width {rw.slope:.4f}, area {ra.slope:.4f}")
print(f"  length grows {rl.slope / rw.slope:.2f}× faster than width "
      f"(t = {t_rw:.2f}, p = {2 * stats.t.sf(abs(t_rw), 2 * len(dai) - 4):.1e})")
print(f"  area doubling time = {np.log(2) / ra.slope:.1f} days\n")

# Which model describes the growth of discolored area best? Same rule as Cell 3: lowest AICc wins,
# and only the best model is reported.
y_a = les["Area_cm2"].to_numpy(float)
area_models = {"Exponential": (lambda t, a, r: a * np.exp(r * t), [0.5, 0.017], ["A0 (cm²)", "r (d⁻¹)"]),
               "Linear":      (lambda t, a, b: a + b * t,         [0.5, 0.01],  ["a (cm²)", "b (cm² d⁻¹)"]),
               "Power":       (lambda t, a, b: a * t ** b,        [1e-3, 1.5],  ["a", "b"])}
rank = []
for name, (f, p0, names) in area_models.items():
    p, cov = curve_fit(f, dai, y_a, p0=p0, maxfev=20000)
    n, k = len(y_a), len(p) + 1                                   # +1 for the error variance
    rss = ((y_a - f(dai, *p)) ** 2).sum()
    rank.append(dict(Model=name, AICc=n * np.log(rss / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1), p=p, cov=cov, f=f, names=names))
rank = pd.DataFrame(rank).sort_values("AICc").reset_index(drop=True)
rank["dAICc"] = rank["AICc"] - rank["AICc"].min()
rank["Weight"] = np.exp(-rank["dAICc"] / 2) / np.exp(-rank["dAICc"] / 2).sum()   # Akaike weight
print("Area: model ranking (lowest AICc = best)")
print(rank[["Model", "AICc", "dAICc", "Weight"]].round(3).to_string(index=False))

best_A = rank.iloc[0]
resid = y_a - best_A["f"](dai, *best_A["p"])
ci = stats.t.ppf(0.975, len(y_a) - len(best_A["p"])) * np.sqrt(np.diag(best_A["cov"]))
print(f"-> Best model: {best_A['Model']}  (R² = {1 - (resid ** 2).sum() / ((y_a - y_a.mean()) ** 2).sum():.3f}, "
      f"residual Shapiro–Wilk p = {stats.shapiro(resid).pvalue:.3f})")
for nm, a, c in zip(best_A["names"], best_A["p"], ci):
    print(f"  {nm:12s} = {a:.4f}   95% CI {a - c:.4f} – {a + c:.4f}")


## 5. Figure 3

In [ ]:
# =====================================================================
# CELL 5 — Figure 3: temperature and pH responses
# =====================================================================
fig, axs = plt.subplots(1, 2, figsize=(W, 4.0), layout="constrained")
# one tuple per panel: axis, data, x column, group stats, best model (from Cell 3), labels, x-range
panels = [(axs[0], temp, "Temperature_C", st_t, best_T, "Temperature (°C)", "(a) Temperature", (7, 43)),
          (axs[1], ph, "pH", st_p, best_p, "pH", "(b) pH", (2.5, 8.5))]
for ax, df, x, s, m, xl, title, lim in panels:
    f, p, lo, hi = m["f"], m["p"], m["lo"], m["hi"]              # best model and 95% CI of its optimum
    ax.axvspan(lo, hi, color=ORANGE, alpha=0.18, lw=0, label="Optimum (95% CI)")
    for b in (lo, hi):
        ax.axvline(b, color=ORANGE, ls="--", lw=0.8)
    xs = np.linspace(*lim, 400)
    xs = xs[f(xs, *p) > 0]                                       # draw the curve only where growth > 0
    ax.plot(xs, f(xs, *p), color=RED, lw=1.4, label="Best-fit model")
    jit = rng.uniform(-1, 1, len(df)) * (0.45 if x == "Temperature_C" else 0.06)   # small horizontal jitter
    ax.scatter(df[x] + jit, df["Radial_Growth_mm"], s=16, color=BLUE, edgecolor="black", lw=0.3, alpha=0.8, label="Replicate", zorder=3)
    ax.errorbar(s.index, s["mean"], yerr=s["se"], fmt="o", ms=5, color=NAVY, mec="black", mew=0.4, label="Mean ± SE", zorder=4, **EB)
    for xi, (mu, se, l) in zip(s.index, s[["mean", "se", "letter"]].to_numpy()):   # significance letters
        ax.text(xi, mu + se + 4, l, ha="center", va="bottom", fontsize=VAL, bbox=dict(fc="white", ec="none", pad=0.5))
    ax.set(xlim=lim, ylim=(0, 115), xlabel=xl, ylabel="Colony diameter (mm)", title=title)
axs[0].set_xticks([10, 15, 20, 25, 30, 35, 40])
axs[1].set_xticks([3, 4, 5, 5.5, 6, 7, 8], ["3", "4", "5", "5.5", "6", "7", "8"])
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="outside lower center", ncol=4)             # one shared legend below both panels
fig.suptitle("Effect of temperature and pH on mycelial growth of isolate R144")
save(fig, "Figure3_temperature_pH")
print(f"Figure 3 drawn with: temperature = {best_T['name']} model, pH = {best_p['name']} model")
print(f"  shaded band (a): {best_T['lo']:.2f}–{best_T['hi']:.2f} °C   shaded band (b): pH {best_p['lo']:.2f}–{best_p['hi']:.2f}")


## 6. Figure 5

In [ ]:
# =====================================================================
# CELL 6 — Figure 5: extracellular enzyme activities
# =====================================================================
se_ = st_e.sort_values("mean", ascending=False)                  # highest activity on the left
colors = [BLUE, RED, GREEN, ORANGE, PURPLE, NAVY]
fig, ax = plt.subplots(figsize=(W, 3.8), constrained_layout=True)
xp = np.arange(len(se_))
ax.bar(xp, se_["mean"], width=0.6, color=colors, edgecolor="black", lw=0.4, zorder=2)
ax.errorbar(xp, se_["mean"], yerr=se_["se"], fmt="none", zorder=3, **EB)
for i, e in enumerate(se_.index):
    v = enz.loc[enz["Enzyme"] == e, "Activity_U_per_ml"].to_numpy()
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=14, color="white", edgecolor="black", lw=0.4, zorder=4)  # replicates
    top = max(v.max(), se_.loc[e, "mean"] + se_.loc[e, "se"])
    ax.text(i, top + 0.4, f"{se_.loc[e, 'mean']:.2f} {se_.loc[e, 'letter']}", ha="center", va="bottom", fontsize=VAL)
ax.set_xticks(xp, se_.index)
ax.set(ylim=(0, 18), ylabel="Activity (U mL$^{-1}$)", title="Extracellular enzyme activities of isolate R144")
ax.grid(axis="x", visible=False)
save(fig, "Figure5_enzymes")
# Text summary under the figure: mean ± SE, Tukey letter, CV and fold difference vs the lowest enzyme
sd_ = enz.groupby("Enzyme")["Activity_U_per_ml"].std()
summary = se_[["mean", "se", "letter"]].copy()
summary["CV_%"] = 100 * sd_[summary.index] / summary["mean"]
summary["fold_vs_lowest"] = summary["mean"] / summary["mean"].min()
print(summary.round(2).to_string())


## 7. Figure 6

In [ ]:
# =====================================================================
# CELL 7 — Figure 6: development of discolored zones
# =====================================================================
dai_col = {60: BLUE, 120: ORANGE, 180: RED}                      # same colour per harvest in every panel
spec = [("Length_cm", "(a) Length", "Length (cm)"), ("Width_cm", "(b) Width", "Width (cm)"),
        ("Area_cm2", "(c) Area", "Area (cm$^2$)"), ("LW_ratio", "(d) Length:width ratio", "Length:width ratio")]
fig, axs = plt.subplots(3, 2, figsize=(W, 9.4), constrained_layout=True)

# Panels (a)–(d): bar = mean, error bar = SE, open circles = individual zones, text = mean + letter
for ax, (v, title, yl) in zip(axs.flat[:4], spec):
    s = st_l[v]
    xp = np.arange(len(s))
    ax.bar(xp, s["mean"], width=0.55, color=[dai_col[d] for d in s.index], edgecolor="black", lw=0.4, zorder=2)
    ax.errorbar(xp, s["mean"], yerr=s["se"], fmt="none", zorder=3, **EB)
    for i, d in enumerate(s.index):
        vals = les.loc[les["DAI"] == d, v].to_numpy()
        ax.scatter(i + rng.uniform(-0.13, 0.13, len(vals)), vals, s=10, color="white", edgecolor="black", lw=0.4, zorder=4)
        top = max(vals.max(), s.loc[d, "mean"] + s.loc[d, "se"])
        ax.text(i, top + 0.04 * les[v].max(), f"{s.loc[d, 'mean']:.2f} {s.loc[d, 'letter']}", ha="center", va="bottom", fontsize=VAL)
    ax.set_xticks(xp, [str(d) for d in s.index])
    ax.set(ylim=(0, les[v].max() * 1.22), xlabel="Days after inoculation", ylabel=yl, title=title)
    ax.grid(axis="x", visible=False)

# Panel (e): length and width on a log scale; straight lines = exponential growth, slope = rate r
ax = axs[2, 0]
tx = np.linspace(55, 185, 200)
for col, r, c, lab in [("Length_cm", rl, PURPLE, "Length"), ("Width_cm", rw, GREEN, "Width")]:
    ax.scatter(dai + rng.uniform(-3, 3, len(dai)), les[col], s=14, color=c, edgecolor="black", lw=0.3, zorder=3)
    ax.plot(tx, np.exp(r.intercept + r.slope * tx), color=c, lw=1.4, label=f"{lab}, r = {r.slope:.4f} d$^{{-1}}$")
ax.set_yscale("log")
ax.yaxis.set_minor_locator(plt.NullLocator())
ax.set_yticks([1, 2, 3, 4, 5, 6], ["1", "2", "3", "4", "5", "6"])
ax.set(xticks=[60, 120, 180], ylim=(0.8, 8), xlabel="Days after inoculation", ylabel="Dimension (cm, log scale)", title="(e) Relative expansion")
ax.legend(loc="upper left")

# Panel (f): area with the best-fit model from Cell 4
ax = axs[2, 1]
ax.scatter(dai + rng.uniform(-3, 3, len(dai)), les["Area_cm2"], s=14, color=BLUE, edgecolor="black", lw=0.3, zorder=3)
ax.plot(tx, best_A["f"](tx, *best_A["p"]), color=RED, lw=1.4, label=f"{best_A['Model']} model")   # best model only (Cell 4)
ax.set(xticks=[60, 120, 180], ylim=(0, 18), xlabel="Days after inoculation", ylabel="Area (cm$^2$)", title="(f) Area kinetics")
ax.legend(loc="upper left")
fig.suptitle("Development of discolored zones after inoculation with isolate R144")
save(fig, "Figure6_discolored_zones")
# Text summary under the figure: mean ± SE and Tukey/Games–Howell letter at each harvest
for v in ["Length_cm", "Width_cm", "Area_cm2", "LW_ratio"]:
    s = st_l[v]
    print(f"{v:10s}  " + "   ".join(f"{d} DAI: {s.loc[d, 'mean']:.2f} ± {s.loc[d, 'se']:.2f} {s.loc[d, 'letter']}" for d in s.index))
print(f"Panel (e) rates: length r = {rl.slope:.4f} d⁻¹, width r = {rw.slope:.4f} d⁻¹;  panel (f): {best_A['Model']} model")


## 8. Download

In [ ]:
# =====================================================================
# CELL 8 — Download all figures as one zip file
# =====================================================================
shutil.make_archive("R144_figures", "zip", OUT)
print("Files in the zip:")
for f in sorted(os.listdir(OUT)):
    print("  ", f)
files.download("R144_figures.zip")
